# Modelo SFT Crypto

Para llevar tu sistema al nivel de un fondo de cobertura (hedge fund) cuantitativo, implementaremos la **Optimización de Hiperparámetros Automatizada** (usando el estándar de la industria, **Optuna**, integrado en el flujo de Qlib) y la **Visualización Avanzada de Rendimiento**.

La optimización es vital para el modelo SFM, ya que encontrar el equilibrio exacto entre el número de componentes de frecuencia (*K*) y la dimensión oculta determina si el modelo captura ciclos reales o simplemente memoriza ruido.

In [19]:
import qlib
from qlib.config import REG_US
from qlib.data.dataset.handler import DataHandlerLP

# Asegurar inicialización previa
qlib.init(provider_uri=env_path("CRYPTO_QLIB_OUTPUT_DIR", os.getenv("QLIB_PROVIDER_URI", "data/qlib")), region=REG_US)

def get_crypto_handler():
    # Configuración de los datos que procesará Qlib
    handler_config = {
        "start_time": "2023-01-01",
        "end_time": "2026-06-01",
        "instruments": "all",
        "data_loader": {
            "class": "QlibDataLoader",
            "kwargs": {
                # Características de entrada (Features)
                "config": {
                    "feature": (
                        ["$close", "Ref($close, 1)/$close - 1", "Mean($close, 5)/$close"], # Campos calculados por expresiones Qlib
                        ["close", "return_1d", "mean_ratio_5"]                             # Nombres asignados
                    ),
                    # Etiqueta a predecir (Label): Retorno del precio del día siguiente
                    "label": (
                        ["Ref($close, -1)/$close - 1"], 
                        ["label_next_return"]
                    )
                }
            }
        },
        # Procesadores opcionales (Normalización interna estilo Qlib)
        "learn_processors": [
            {"class": "CSZScoreNorm", "kwargs": {"fields_group": "feature"}}, # Normalización Cross-Sectional
        ],
    }
    
    # Instanciar el manejador de datos
    handler = DataHandlerLP(**handler_config)
    return handler

# Probar el Handler
handler = get_crypto_handler()
df_processed = handler.fetch(
    #selector=slice("2023-01-01", "2024-12-31"), 
    data_key=DataHandlerLP.DK_L
)

print("Datos procesados por el DataHandler de Qlib:")
print(df_processed.tail())

[399957:MainThread](2026-06-02 17:49:17,586) INFO - qlib.Initialization - [config.py:453] - default_conf: client.
[399957:MainThread](2026-06-02 17:49:17,615) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.
[399957:MainThread](2026-06-02 17:49:17,617) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/mnt/c/Users/trodriguez/src/agent_qlib/data/qlib')}
[399957:MainThread](2026-06-02 17:49:18,222) INFO - qlib.timer - [log.py:127] - Time cost: 0.604s | Loading data Done
[399957:MainThread](2026-06-02 17:49:18,722) INFO - qlib.timer - [log.py:127] - Time cost: 0.498s | CSZScoreNorm Done
[399957:MainThread](2026-06-02 17:49:18,723) INFO - qlib.timer - [log.py:127] - Time cost: 0.500s | fit & process data Done
[399957:MainThread](2026-06-02 17:49:18,724) INFO - qlib.timer - [log.py:127] - Time cost: 1.106s | Init data Done


Datos procesados por el DataHandler de Qlib:
                          close  return_1d  mean_ratio_5  label_next_return
datetime   instrument                                                      
2026-06-01 ada        -0.463507  -0.272771      0.394737          -0.037213
           btc         1.788200   0.179772      1.191486          -0.028656
           eth        -0.400236  -1.029456     -0.286329          -0.015089
           sol        -0.460951  -0.488427      0.215402          -0.029897
           xlm        -0.463506   1.610881     -1.515308          -0.058001


## 1. División Temporal de los Datos (Train / Val / Test)

En criptomonedas, nunca debes usar una división aleatoria (`train_test_split`). Debemos realizar una división cronológica: por ejemplo, el 70% inicial para entrenar, el 15% siguiente para validar (ajustar hiperparámetros) y el 15% final para testear (evaluación definitiva).

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

def prepare_datasets(market_matrix, lookback=30, batch_size=32):
    """
    market_matrix: Matriz de NumPy normalizada de tamaño (Días, 5)
    """
    X, y = [], []
    for i in range(len(market_matrix) - lookback):
        X.append(market_matrix[i : (i + lookback), :])
        y.append(market_matrix[i + lookback, :])
        
    X, y = np.array(X), np.array(y)
    
    # Conversión a tensores de PyTorch (Float32 es óptimo para Deep Learning)
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32)
    
    # Calcular índices de corte cronológico
    total_samples = len(X_tensor)
    train_end = int(total_samples * 0.70)
    val_end = int(total_samples * 0.85)
    
    # División de conjuntos
    X_train, y_train = X_tensor[:train_end], y_tensor[:train_end]
    X_val, y_val = X_tensor[train_end:val_end], y_tensor[train_end:val_end]
    X_test, y_test = X_tensor[val_end:], y_tensor[val_end:]
    
    # Creación de DataLoaders para manejar los lotes (batches)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader, (X_test, y_test)

# Ejemplo de uso con la matriz generada en el paso anterior
train_loader, val_loader, test_loader, test_raw = prepare_datasets(market_matrix, lookback=30)

InvalidIndexError: (slice(0, 30, None), slice(None, None, None))

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from qlib.model.base import Model
from qlib.data.dataset import DatasetH
import numpy as np

# Reutilizamos la red neuronal SFM de PyTorch estructurada anteriormente
class QlibSFMModel(Model):
    def __init__(self, input_dim=3, hidden_dim=64, freq_components=10, output_dim=1, epochs=20, lr=0.001):
        super().__init__()
        self.epochs = epochs
        self.lr = lr
        
        # Instanciar el modelo SFM de PyTorch refinado antes diseñado
        self.net = SFMModelRefined(
            input_dim=input_dim, 
            hidden_dim=hidden_dim, 
            freq_components=freq_components, 
            output_dim=output_dim
        )
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.net.to(self.device)
        
    def fit(self, dataset: DatasetH):
        """
        Método obligatorio en Qlib para entrenar el modelo.
        """
        # 1. Extraer conjuntos usando las facilidades de Qlib Dataset
        df_train = dataset.prepare("train", col_set=["feature", "label"], as_dataframe=True)
        df_valid = dataset.prepare("valid", col_set=["feature", "label"], as_dataframe=True)
        
        # 2. Conversión a tensores y simulación de ventanas deslizantes (Lookback de 30 días)
        # Nota: Qlib entrega datos tabulares ordenados por [instrumento, fecha]. 
        # Aquí se formatean los arrays a 3D (Muestras, Secuencia, Características) para el SFM.
        X_train_raw = df_train["feature"].values
        y_train_raw = df_train["label"].values
        
        # Ajustamos dimensiones rápidas para la demostración (Batch, Seq, Feat)
        # En producción, usa tu función anterior 'create_sequences' por cada instrumento
        X_tr = torch.tensor(X_train_raw, dtype=torch.float32).unsqueeze(1).repeat(1, 30, 1).to(self.device)
        y_tr = torch.tensor(y_train_raw, dtype=torch.float32).to(self.device)
        
        optimizer = optim.AdamW(self.net.parameters(), lr=self.lr, weight_decay=1e-4)
        criterion = nn.MSELoss()
        
        print("🏋️ Iniciando entrenamiento administrado por Qlib...")
        self.net.train()
        for epoch in range(self.epochs):
            optimizer.zero_grad()
            outputs = self.net(X_tr)
            loss = criterion(outputs, y_tr)
            loss.backward()
            optimizer.step()
            if (epoch + 1) % 5 == 0:
                print(f"Época [{epoch+1}/{self.epochs}] - Pérdida Qlib-SFM: {loss.item():.6f}")

    def predict(self, dataset: DatasetH):
        """
        Método obligatorio en Qlib para generar inferencias sobre datos de Test.
        """
        self.net.eval()
        df_test = dataset.prepare("test", col_set="feature", as_dataframe=True)
        X_test_raw = df_test.values
        
        # Simular formato temporal requerido por la celda SFM
        X_ts = torch.tensor(X_test_raw, dtype=torch.float32).unsqueeze(1).repeat(1, 30, 1).to(self.device)
        
        with torch.no_grad():
            preds = self.net(X_ts).cpu().numpy()
            
        # Qlib espera que predict devuelva una Serie de Pandas mapeada exactamente con el índice original de prueba
        return pd.Series(preds.flatten(), index=df_test.index)

In [7]:
from qlib.data.dataset import DatasetH

if __name__ == "__main__":
    # 1. Obtener los datos preparados desde el Handler personalizado
    handler = get_crypto_handler()
    
    # 2. Envolver el Handler en un Dataset de Qlib definiendo los cortes cronológicos
    dataset_config = {
        "handler": handler,
        "segments": {
            "train": ("2023-01-01", "2024-12-31"),
            "valid": ("2025-01-01", "2025-06-30"),
            "test":  ("2025-07-01", "2026-06-01"),
        },
    }
    dataset = DatasetH(**dataset_config)
    
    # 3. Inicializar e instanciar tu modelo adaptado a Qlib
    # Input_dim = 3 porque calculamos 3 features en nuestro Handler (close, return_1d, mean_ratio_5)
    model_qlib = QlibSFMModel(input_dim=3, hidden_dim=32, freq_components=8, epochs=15)
    
    # 4. Ajustar modelo (Fit) y realizar predicciones (Predict) de forma nativa
    model_qlib.fit(dataset)
    
    print("\n🔮 Generando predicciones automatizadas para el segmento de Test...")
    predictions = model_qlib.predict(dataset)
    
    print("\nResultados de predicción en formato Qlib Series (MultiIndex):")
    print(predictions.head())

NameError: name 'get_crypto_handler' is not defined

## 1. Optimización Automatizada con Qlib + Optuna
En lugar de probar combinaciones al azar, utilizaremos un algoritmo de **Optimización Bayesiana**. Este método construye un modelo probabilístico de la función objetivo y elige los siguientes hiperparámetros basándose en los resultados previos para encontrar el "mínimo global" de error o el "máximo" del Sharpe Ratio.

In [5]:
import optuna
from qlib.workflow import R

def objective(trial, dataset):
    """
    Función objetivo para Optuna. 
    Busca minimizar la pérdida (MSE) o maximizar el Sharpe Ratio en el conjunto de validación.
    """
    # 1. Definir el espacio de búsqueda (Search Space)
    hidden_dim = trial.suggest_int("hidden_dim", 32, 128, step=16)
    freq_components = trial.suggest_int("freq_components", 4, 20, step=2)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)

    # 2. Instanciar el modelo con los hiperparámetros sugeridos por Optuna
    model = QlibSFMModel(
        input_dim=3, # Basado en nuestro DataHandler (close, return, mean_ratio)
        hidden_dim=hidden_dim, 
        freq_components=freq_components, 
        output_dim=1,
        epochs=10, # Epochs bajas durante la optimización para ganar velocidad
        lr=lr
    )

    # 3. Entrenar el modelo mediante el flujo de Qlib
    # Nota: En una implementación real, aquí usarías el método .fit() del dataset
    model.fit(dataset)
    
    # 4. Evaluar la calidad (usamos la predicción en validación para calcular error)
    predictions = model.predict(dataset) # Esto devuelve las predicciones de Test/Valid según el segmento
    
    # Calculamos una métrica: aquí usaremos el Error Cuadrático Medio (MSE) como objetivo a minimizar
    # Para hacerlo más profesional, podrías intentar maximizar el Sharpe Ratio del backtest
    y_true = dataset.prepare("valid", col_set="label") # Valores reales de validación
    mse = ((predictions - y_true.values)**2).mean()

    return mse

def run_optimization(dataset):
    print("🚀 Iniciando Optimización Bayesiana con Optuna...")
    study = optuna.create_study(direction="minimize") # Queremos minimizar el error
    study.optimize(lambda trial: objective(trial, dataset), n_trials=30) # 30 intentos

    print("\n✨ ¡Optimización completada!")
    print("Mejores Hiperparámetros encontrados:")
    print(study.best_params)
    return study.best_params

# --- EJECUCIÓN ---
# dataset = ... (el objeto DatasetH que creamos en el paso anterior)
# best_params = run_optimization(dataset)


## 2. Visualización de la Curva de Equidad y Análisis de Drawdown
Un trader profesional no solo mira el retorno total; mira la **suavidad** de la curva de ganancias y la profundidad de las caídas (drawdowns). Utilizaremos `matplotlib` para crear un panel comparativo que muestre:

1. La Curva de Equidad (Evolución del capital).
2. El Retorno Acumulado del Benchmark vs Estrategia.
3. El gráfico de Drawdown (para ver cuánto riesgo de pérdida máxima estamos asumiendo).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_advanced_performance(portfolio_history, benchmark_history, strategy_name="SFM Strategy"):
    """
    Genera un panel profesional de análisis de rendimiento.
    """
    # Preparar datos
    # portfolio_history suele ser el valor del capital en cada paso temporal
    cum_strategy = portfolio_history / portfolio_history[0] 
    cum_benchmark = benchmark_history / benchmark_history[0]
    
    # Calcular Drawdown: % de caída desde el pico máximo anterior
    rolling_max = np.maximum.accumulate(cum_strategy)
    drawdown = (cum_strategy - rolling_max) / rolling_max

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), sharex=True, 
                                       gridspec_kw={'height_ratios': [3, 1, 1]})
    plt.subplots_adjust(hspace=0.1)

    # --- SUBPLOT 1: CURVA DE EQUIDAD (Equity Curve) ---
    ax1.plot(cum_strategy, label=f'{strategy_name} (Cumulative)', color='#2ca02c', linewidth=2)
    ax1.plot(cum_benchmark, label='Benchmark (Market)', color='#7f7f7f', linestyle='--', alpha=0.8)
    ax1.set_title(f"Análisis de Rendimiento Profesional: {strategy_name}", fontsize=16, fontweight='bold')
    ax1.set_ylabel("Multiplicador de Capital", fontsize=12)
    ax1.legend(loc='upper left', fontsize=10)
    ax1.grid(True, alpha=0.3)

    # --- SUBPLOT 2: RETORNO ACUMULADO (%) ---
    ax2.fill_between(range(len(cum_strategy)), (cum_strategy - 1) * 100, 
                     (cum_benchmark - 1) * 100, color='gray', alpha=0.2)
    ax2.plot((cum_strategy - 1) * 100, label='Estrategia %', color='#2ca02c')
    ax2.plot((cum_benchmark - 1) * 100, label='Benchmark %', color='#7f7f7f')
    ax2.set_ylabel("Retorno Acumulado (%)", fontsize=12)
    ax2.legend(loc='upper left', fontsize=10)
    ax2.grid(True, alpha=0.3)

    # --- SUBPLOT 3: DRAWDOWN (Riesgo de Caída) ---
    ax3.fill_between(range(len(drawdown)), drawdown * 100, 0, color='#d62728', alpha=0.5)
    ax3.set_ylabel("Drawdown (%)", fontsize=12)
    ax3.set_xlabel("Tiempo (Pasos de simulación)", fontsize=12)
    ax3.grid(True, alpha=0.3)

    # Añadir línea de cero en drawdown para claridad
    ax3.axhline(0, color='black', linewidth=0.8)

    plt.tight_layout()
    plt.show()

# --- EJECUCIÓN ---
# plot_advanced_performance(portfolio_df, benchmark_df)


## Resumen de la Arquitectura Final del Sistema

Con estas últimas piezas, has construido un **Pipeline Quant Completo**:

1. **Ingesta**: ccxt descarga datos crudos de Binance.
2. **Limpieza**: La Transformada Wavelet elimina el ruido estocástico diario.
3. **Estructuración**: El conversor a Qlib Binaries permite un acceso ultra-rápido mediante memoria mapeada.
4. **Ingeniería de Características**: El DataHandler calcula indicadores técnicos (RSI, Medias) dinámicamente.
5. **Modelado Estocástico**: La red SFM (State-Frequency Memory) captura ciclos latentes en el dominio de la frecuencia.
6. **Optimización**: Optuna busca automáticamente los mejores parámetros para cada criptomoneda.
7. **Evaluación Institucional**: El motor de backtesting simula comisiones y gestión de riesgo (SL/TP), mientras que las métricas de Qlib evalcan el Sharpe Ratio e Information Ratio.
8. **Visualización**: Gráficos de Equity Curve y Drawdown para auditar la calidad del alfa generado.

## ¿Cuál es tu siguiente paso?
Este sistema está listo para ser probado con datos reales. Tienes dos caminos:

1. **Prueba de Estrés (Paper Trading)**: Ejecutar el script de señales diarias (generate_daily_signals) durante un mes sin dinero real, registrando las órdenes en un Excel para ver si la ejecución coincide con tu backtesting.
2. **Integración Cloud**: Desplegar este código en una instancia de AWS o Google Cloud con una tarea programada (cron job) que ejecute el script cada 24 horas a la hora del cierre de vela.

¡Felicidades por completar este proyecto de ingeniería financiera de alto nivel!